In [46]:


import os
import pandas as pd
import plotly.express as px
import seaborn as sns
import matplotlib.pyplot as plt

data_folder = '../data/'
path = os.path.join(data_folder)

df = pd.DataFrame()
df2 = pd.DataFrame()
for file in os.listdir(path):
    if 'price_history_full' in file:
        df2 = pd.read_csv(data_folder + file)
        df = pd.concat([df, df2])
        
        

meta_path = '../data/all_asins_meta.csv'
meta_df = pd.read_csv(meta_path)

df = df.merge(meta_df, on = 'asin')

# KEEPA_EPOCH = pd.Timestamp('2011-01-01') #19.05 - - - Not needed anymore, we've already handled it before importing to SQL
# df['listed_since'] = pd.to_datetime(KEEPA_EPOCH + pd.to_timedelta(df['listed_since'], 'minutes'))

df['datetime'] = pd.to_datetime(df['datetime'], utc = True)
df['listed_since'] = pd.to_datetime(df['listed_since'], utc = True)
df['days_since_launch'] = (df['datetime'] - df['listed_since']).dt.days
df = df[df['days_since_launch'] > 0]
print(df.shape)

df['NEW'] = df['new_price'] * 100 #had to implement this here in the new pipeline, as it wasn't handled
df = df.drop('new_price', axis = 1)

#new price I'd like to check - amazon - maybe it will give some nice insights as well
df['AMAZON'] = df['amazon'] * 100
df = df.drop('amazon', axis = 1)

df['median'] = df.groupby('asin')['NEW'].transform('median')
df = df[df['NEW'] < 3 * df['median']]
df = df[df['NEW'].notna()]

df = df.set_index('datetime', drop = True)
backup_df = df
df = df.groupby('asin').resample('W')['NEW'].min().reset_index()
df['NEW'] = df.groupby('asin')['NEW'].ffill()


df = df.merge(meta_df[['asin', 'brand', 'submodel_name', 'generation_name', 'listed_since']], on='asin', how='left')
df = df.rename(columns = {'generation_name': 'model'})

df['listed_since'] = pd.to_datetime(df['listed_since'], utc = True)
df['days_since_launch'] = (df['datetime'] - df['listed_since']).dt.days


'''Computing first price, price_pct_of_launch etc'''

df['first_price'] = df.groupby('submodel_name')['NEW'].transform('first')

df['price_pct_of_launch'] = round(df['NEW'] / df['first_price'] * 100, 1)
df['days_rounded'] = (df['days_since_launch'] / 7).round() * 7

df.head()

(878170, 26)


,asin,datetime,NEW,brand,submodel_name,model,listed_since,days_since_launch,first_price,price_pct_of_launch,days_rounded
0,B011SDYBZW,2021-07-04 00:00:00+00:00,569.0,Apple,iPhone 11,iPhone 11,2018-12-19 00:00:00+00:00,928,569.0,100.0,931.0
1,B011SDYBZW,2021-07-11 00:00:00+00:00,551.0,Apple,iPhone 11,iPhone 11,2018-12-19 00:00:00+00:00,935,569.0,96.8,938.0
2,B011SDYBZW,2021-07-18 00:00:00+00:00,551.0,Apple,iPhone 11,iPhone 11,2018-12-19 00:00:00+00:00,942,569.0,96.8,945.0
3,B011SDYBZW,2021-07-25 00:00:00+00:00,529.0,Apple,iPhone 11,iPhone 11,2018-12-19 00:00:00+00:00,949,569.0,93.0,952.0
4,B011SDYBZW,2021-08-01 00:00:00+00:00,529.0,Apple,iPhone 11,iPhone 11,2018-12-19 00:00:00+00:00,956,569.0,93.0,959.0


In [47]:

df['tier'] = df.apply(
    lambda row: row['submodel_name'].replace(row['model'], '').strip(),
    axis=1
)
df['tier'] = df['tier'].replace('', 'Base')



ms_df = pd.read_csv('../data/monthly_sold_full.csv')
ms_df['datetime'] = pd.to_datetime(ms_df['datetime'])
ms_df['premiere_date'] = pd.to_datetime(ms_df['premiere_date'])
ms_df['days_since_launch'] = (ms_df['datetime'] - ms_df['premiere_date']).dt.days
ms_df['days_rounded'] = (ms_df['days_since_launch'] / 7).round() * 7
ms_df = ms_df[ms_df['days_since_launch'] >= 0]


ms_df['tier'] = ms_df.apply(
    lambda row: row['submodel_name'].replace(row['generation_name'], '').strip(),
    axis=1
)
ms_df['tier'] = ms_df['tier'].replace('', 'Base')
print(ms_df['tier'].value_counts())
ms_df.head()



tier
Base        6005
Pro         3957
Pro Max     2309
Ultra       1577
a           1053
FE           965
Mini         781
Plus         737
+            730
Pro Fold      41
Edge          32
Pro XL        24
Name: count, dtype: int64


,asin,product_grade,submodel_name,brand,generation_name,premiere_date,datetime,monthly_sold,days_since_launch,days_rounded,tier
0,B011SDYBZW,Renewed Premium,iPhone 11,Apple,iPhone 11,2019-09-20,2023-10-20 07:04:00,50,1491,1491.0,Base
1,B011SDYBZW,Renewed Premium,iPhone 11,Apple,iPhone 11,2019-09-20,2023-11-28 13:34:00,100,1530,1533.0,Base
2,B011SDYBZW,Renewed Premium,iPhone 11,Apple,iPhone 11,2019-09-20,2024-01-01 08:42:00,50,1564,1561.0,Base
3,B011SDYBZW,Renewed Premium,iPhone 11,Apple,iPhone 11,2019-09-20,2024-01-01 18:16:00,100,1564,1561.0,Base
4,B011SDYBZW,Renewed Premium,iPhone 11,Apple,iPhone 11,2019-09-20,2024-01-02 14:56:00,50,1565,1568.0,Base


In [48]:
#W4 D1 T3

'''
Filter `df` to tier == 'Base' (or a tier of your choice that exists across multiple brands).
Groupby `generation_name + days_rounded` → mean `price_pct_of_launch`.
Plot Plotly line: x = days_rounded, y = price_pct_of_launch, color = generation_name.
Add hline at y=50. Title: "Price Decay — [tier] models across generations".

Use Plotly legend toggles to explore — click individual generations on/off.
Note any clear pattern: do newer generations hold value better than older ones?'''


base_df = df[df['tier'] == 'Base']
base_df.head()

grouped_base_df = base_df.groupby(['model', 'days_rounded'])['price_pct_of_launch'].agg('mean').reset_index()
grouped_base_df.head()
line_plot = px.line(
    grouped_base_df,
    x = 'days_rounded',
    y = 'price_pct_of_launch',
    color = 'model',
    title = 'Price decay - [tier] models across generations'
)
line_plot.add_hline(y = 50)
line_plot.show()

In [49]:
#W4 D1 T4
'''Pick one generation with multiple tiers (e.g. iPhone 13 — has Base, Mini, Pro, Pro Max).
Filter `df` to that generation_name.
Groupby `submodel_name + days_rounded` → mean `price_pct_of_launch`.
Plot Plotly line: x = days_rounded, y = price_pct_of_launch, color = submodel_name.
Add hline at y=50. Title: "Price Decay — [generation] by submodel".

Does the Pro Max hold value better than the base model? Does the Mini decay faster?'''

generation_df = df[df['model'] == 'iPhone 13']
grouped_generation_decay_df = generation_df.groupby(['submodel_name', 'days_rounded'])['price_pct_of_launch'].agg('mean').reset_index()
print(generation_df['first_price'].nunique())

grouped_generation_decay_chart = px.line(
    grouped_generation_decay_df,
    x = 'days_rounded',
    y = 'price_pct_of_launch',
    color = 'submodel_name',
    title = 'Price decay - [generation] by submodel'
)
grouped_generation_decay_chart.add_hline(y = 50)
grouped_generation_decay_chart.show()

4


In [50]:
df[df['model'] == 'iPhone 13'].groupby('submodel_name').agg(
    first_date=('datetime', 'first'),
    first_price=('NEW', 'first'),
    premiere_date=('listed_since', 'first')
).assign(days_gap=lambda x: (x['first_date'] - x['premiere_date']).dt.days)


,first_date,first_price,premiere_date,days_gap
submodel_name,,,,
iPhone 13,2022-03-06 00:00:00+00:00,721.00,2021-11-11 19:56:00+00:00,114
iPhone 13 Mini,2021-12-12 00:00:00+00:00,689.99,2021-11-10 22:36:00+00:00,31
iPhone 13 Pro,2022-06-12 00:00:00+00:00,1399.00,2021-11-11 21:48:00+00:00,212
iPhone 13 Pro Max,2022-04-24 00:00:00+00:00,1649.97,2021-11-11 22:40:00+00:00,163
